In [1]:
pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

def analyze_parquet(file_path):
    try:
        # 1. Wczytanie pliku
        df = pd.read_parquet(file_path)
        
        print(f"--- Podstawowa analiza pliku: {file_path} ---")
        
        # 2. Wyświetlenie pierwszych kilku wierszy
        print("\n[Podgląd danych - pierwsze 5 wierszy]:")
        print(df.head())
        
        # 3. Informacje o strukturze (typy kolumn, liczba niepustych rekordów)
        print("\n[Informacje o strukturze i typach danych]:")
        print(df.info())
        
        # 4. Podstawowe statystyki opisowe (dla kolumn numerycznych)
        print("\n[Statystyki opisowe]:")
        print(df.describe())
        
        # 5. Sprawdzenie brakujących danych
        print("\n[Liczba brakujących wartości w kolumnach]:")
        print(df.isnull().sum())
        
        # 6. Wymiary ramki danych
        print(f"\nRozmiar zbioru: {df.shape[0]} wierszy i {df.shape[1]} kolumn.")

    except Exception as e:
        print(f"Wystąpił błąd podczas otwierania pliku: {e}")

# Uruchomienie funkcji
analyze_parquet('data_ub.parquet')

--- Podstawowa analiza pliku: data_ub.parquet ---

[Podgląd danych - pierwsze 5 wierszy]:
       OND          MARKET LANGUAGE  \
0  CEG-CAJ  GREAT MERIDIAN  LANG_03   
1  CAJ-BIF  SOUTH MERIDIAN  LANG_09   
2  CEW-BIW    GREAT AURORA  LANG_10   
3  BIC-CAJ      UPPER APEX  LANG_09   
4  CUN-CAJ   NORTH HORIZON  LANG_04   

                                            segments recommendations  \
0  [{"ORIGIN_AIRPORT_CODE":"CEG","DESTINATION_AIR...            [""]   
1  [{"ORIGIN_AIRPORT_CODE":"CAJ","DESTINATION_AIR...            [""]   
2  [{"ORIGIN_AIRPORT_CODE":"CEW","DESTINATION_AIR...            [""]   
3  [{"ORIGIN_AIRPORT_CODE":"BIC","DESTINATION_AIR...            [""]   
4  [{"ORIGIN_AIRPORT_CODE":"CUN","DESTINATION_AIR...            [""]   

                                          emds_order TRIP_START_DATE  \
0  ["MEAL","FAST TRACK","BAGGAGE","SPECIAL EQUIPM...      2038-01-31   
1  ["BUSINESS LOUNGE","FAST TRACK","PET","BAGGAGE...      2038-02-04   
2  ["BAGGAGE","MEAL","FAST

In [3]:
import pandas as pd
import json
import numpy as np

def transform_flight_data(file_path):
    # 1. Wczytanie danych
    print("Wczytywanie danych...")
    df = pd.read_parquet(file_path)
    
    # 2. Konwersja dat
    print("Konwersja dat...")
    df['TRIP_START_DATE'] = pd.to_datetime(df['TRIP_START_DATE'], errors='coerce')
    df['SALES_DATE'] = pd.to_datetime(df['SALES_DATE'], errors='coerce')
    
    # Obliczenie własnego DTD na podstawie dat (weryfikacja ujemnych wartości)
    df['CALCULATED_DTD'] = (df['TRIP_START_DATE'] - df['SALES_DATE']).dt.days
    
    # 3. Obsługa braków danych i anomalii
    print("Uzupełnianie braków danych...")
    # Zastąpienie braków w programie lojalnościowym wartością 'No_Program'
    df['LOYALTY_MEMBERSHIP_PROGRAM'] = df['LOYALTY_MEMBERSHIP_PROGRAM'].fillna('No_Program')
    
    # Przeglądarka - brakujące jako 'Unknown'
    df['BROWSER_TYPE'] = df['BROWSER_TYPE'].fillna('Unknown')
    
    # Zastąpienie brakujących godzin medianą
    median_hour = df['HOUR_OF_THE_DAY_PL_TIME'].median()
    df['HOUR_OF_THE_DAY_PL_TIME'] = df['HOUR_OF_THE_DAY_PL_TIME'].fillna(median_hour)
    
    # Usunięcie lub oflagowanie ujemnych DTD (opcjonalnie można usunąć: df = df[df['DTD'] >= 0])
    df['IS_ANOMALY_DTD'] = df['DTD'] < 0
    
    # 4. Tworzenie nowych cech (Feature Engineering)
    print("Tworzenie nowych cech...")
    df['TOTAL_PASSENGERS'] = df['ADULTS'] + df['TEENAGERS'] + df['CHILDREN'] + df['INFANTS']
    df['HAS_CHILDREN_OR_INFANTS'] = ((df['CHILDREN'] > 0) | (df['INFANTS'] > 0)).astype(int)
    
    # 5. Parsowanie kolumn typu JSON (Wyciąganie wartości z EMD_INFO)
    # EMD (Electronic Miscellaneous Document) to usługi dodatkowe np. bagaż, fast track
    print("Parsowanie struktur JSON...")
    
    def extract_emd_value(emd_string):
        """Zlicza łączną kwotę wydaną na usługi dodatkowe w danej rezerwacji"""
        if pd.isna(emd_string) or emd_string == '' or emd_string == '[""]':
            return 0.0
        try:
            # Niektóre stringi mogą być listami w formacie tekstowym
            records = json.loads(emd_string)
            total_price = 0.0
            for record in records:
                if isinstance(record, dict) and 'emd_price' in record:
                    total_price += float(record['emd_price'])
            return total_price
        except (json.JSONDecodeError, ValueError, TypeError):
            return 0.0

    # Przeliczenie dla pierwszych 10 000 wierszy dla testu (lub usuń limit, aby przeliczyć całość)
    # df['TOTAL_EMD_SPEND'] = df['EMD_INFO'].apply(extract_emd_value) # Użyj tego dla całego zbioru
    
    print("\n--- Zakończono transformacje ---")
    print("\nPrzykładowe nowe kolumny:")
    print(df[['TRIP_START_DATE', 'SALES_DATE', 'CALCULATED_DTD', 'TOTAL_PASSENGERS', 'IS_ANOMALY_DTD']].head())
    
    return df


Wczytywanie danych...
Konwersja dat...
Uzupełnianie braków danych...
Tworzenie nowych cech...
Parsowanie struktur JSON...

--- Zakończono transformacje ---

Przykładowe nowe kolumny:
  TRIP_START_DATE SALES_DATE  CALCULATED_DTD  TOTAL_PASSENGERS  IS_ANOMALY_DTD
0      2038-01-31 2037-11-06              86                 3           False
1      2038-02-04 2038-01-25              10                 1           False
2      2038-01-02 2037-12-06              27                 1           False
3      2037-11-16 2037-10-28              19                 1           False
4      2038-02-21 2038-01-27              25                 1           False


In [5]:
import pandas as pd
import json
import numpy as np

# Zakładamy, że masz już ramkę danych z poprzedniego kroku o nazwie `df`
df = transform_flight_data('data_ub.parquet')

def advanced_transform_and_analyze(df):
    print("1. Tworzenie zaawansowanych cech czasowych...")
    # Dzień tygodnia i miesiąc wylotu zdradzają, czy to podróż biznesowa, czy wakacyjna
    df['TRIP_MONTH'] = df['TRIP_START_DATE'].dt.month
    df['TRIP_DAY_OF_WEEK'] = df['TRIP_START_DATE'].dt.dayofweek # 0=Poniedziałek, 6=Niedziela
    df['IS_WEEKEND_FLIGHT'] = df['TRIP_DAY_OF_WEEK'].isin([5, 6]).astype(int)
    
    # Sezonowość - przypisanie do kwartału
    df['TRIP_QUARTER'] = df['TRIP_START_DATE'].dt.quarter

    print("2. Analiza głębokich struktur JSON (Złożoność lotu)...")
    def count_segments(segments_str):
        """Zlicza liczbę segmentów lotu (pozwala wykryć przesiadki)"""
        if pd.isna(segments_str) or segments_str == '[""]':
            return 1 # Domyślnie zakładamy lot bezpośredni
        try:
            segments = json.loads(segments_str)
            return len(segments)
        except (json.JSONDecodeError, TypeError):
            return 1

    df['NUMBER_OF_SEGMENTS'] = df['segments'].apply(count_segments)
    df['IS_DIRECT_FLIGHT'] = (df['NUMBER_OF_SEGMENTS'] == 1).astype(int)

    print("3. Wyliczenie całkowitej wartości koszyka (Total Revenue)...")
    # Zakładając, że poprzednio stworzyliśmy kolumnę 'TOTAL_EMD_SPEND'
    # Jeśli nie, wstawiamy 0 jako zabezpieczenie do tego przykładu
    if 'TOTAL_EMD_SPEND' not in df.columns:
        df['TOTAL_EMD_SPEND'] = 0.0 
        
    df['TOTAL_REVENUE'] = df['FARE_YQ_TICKET'] + df['TOTAL_EMD_SPEND']

    print("4. Kategoryzacja okna rezerwacyjnego (Booking Window)...")
    # Zamiast patrzeć na surowe DTD, tworzymy kategorie biznesowe
    bins = [-np.inf, 0, 7, 14, 30, 90, np.inf]
    labels = ['Error/Past', 'Last Minute (0-7)', 'Short (8-14)', 'Medium (15-30)', 'Advance (31-90)', 'Early Bird (90+)']
    df['BOOKING_WINDOW'] = pd.cut(df['CALCULATED_DTD'], bins=bins, labels=labels)

    print("\n--- PODSUMOWANIE ANALITYCZNE ---")
    
    # A. Które rynki generują najwyższy średni przychód?
    print("\n[Top 5 Rynków wg Średniego Przychodu]:")
    market_revenue = df.groupby('MARKET')['TOTAL_REVENUE'].mean().sort_values(ascending=False).head(5)
    print(market_revenue)

    # B. Zależność: Typ lotu vs Przychód i Dni do wylotu
    print("\n[Analiza Typu Lotu (One-Way vs Round-Trip)]:")
    flight_type_analysis = df.groupby('FLIGHT_TYPE').agg(
        Średni_Przychód=('TOTAL_REVENUE', 'mean'),
        Średnie_DTD=('CALCULATED_DTD', 'mean'),
        Liczba_Rezerwacji=('OND', 'count')
    ).round(2)
    print(flight_type_analysis)

    # C. Skuteczność kanałów marketingowych (First Touch)
    print("\n[Konwersja kanałów marketingowych (Wolumen rezerwacji)]:")
    channel_analysis = df['FIRST_TOUCH_CHANNEL'].value_counts(normalize=True).head(5) * 100
    print(channel_analysis.round(2).astype(str) + ' %')

    return df

# Uruchomienie (zakładając, że przekazujesz przetworzony wcześniej DataFrame)
df_advanced = advanced_transform_and_analyze(df)

Wczytywanie danych...
Konwersja dat...
Uzupełnianie braków danych...
Tworzenie nowych cech...
Parsowanie struktur JSON...

--- Zakończono transformacje ---

Przykładowe nowe kolumny:
  TRIP_START_DATE SALES_DATE  CALCULATED_DTD  TOTAL_PASSENGERS  IS_ANOMALY_DTD
0      2038-01-31 2037-11-06              86                 3           False
1      2038-02-04 2038-01-25              10                 1           False
2      2038-01-02 2037-12-06              27                 1           False
3      2037-11-16 2037-10-28              19                 1           False
4      2038-02-21 2038-01-27              25                 1           False
1. Tworzenie zaawansowanych cech czasowych...
2. Analiza głębokich struktur JSON (Złożoność lotu)...
3. Wyliczenie całkowitej wartości koszyka (Total Revenue)...
4. Kategoryzacja okna rezerwacyjnego (Booking Window)...

--- PODSUMOWANIE ANALITYCZNE ---

[Top 5 Rynków wg Średniego Przychodu]:
MARKET
SOUTH ZENITH      173.465594
EAST ZENITH   